In [2]:
import pandas as pd
import requests
import time

headers = {
    'User-Agent': 'WikidataResearch/1.0 (academic research; 1823315676@qq.com)'
}

# ─────────────────────────────────────────────
# Step 1: Read entity list
# ─────────────────────────────────────────────
df = pd.read_csv("sample_15000.csv")
df['QID'] = df['item'].str.extract(r'(Q\d+)')
df = df.drop_duplicates(subset=['QID']).reset_index(drop=True)
qids = df['QID'].dropna().tolist()
print(f"{len(qids)} entities in total")


# ─────────────────────────────────────────────
# Step 2：Fetch complete entity data（wbgetentities）
# ─────────────────────────────────────────────
def get_entities_batch(qid_list):
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbgetentities",
        "ids": "|".join(qid_list),
        "format": "json",
        "languages": "en",
        "props": "labels|claims"
    }
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=60)
        return resp.json()
    except Exception as e:
        print(f"  Request failed: {e}")
        return None

all_entities = {}
batch_size = 50
batches = [qids[i:i+batch_size] for i in range(0, len(qids), batch_size)]
print(f"{len(batches)} batches. Fetching entity data...")

for i, batch in enumerate(batches):
    if i % 20 == 0:
        print(f"  {i+1}/{len(batches)} batch...")
    data = get_entities_batch(batch)
    if data and 'entities' in data:
        all_entities.update(data['entities'])
    time.sleep(0.3)

print(f"\nSuccessfully fetched {len(all_entities)} entities")

15000 entities in total
300 batches，start getting entity data...
  1/300 batch...
  21/300 batch...
  41/300 batch...
  61/300 batch...
  81/300 batch...
  101/300 batch...
  121/300 batch...
  141/300 batch...
  161/300 batch...
  181/300 batch...
  201/300 batch...
  221/300 batch...
  241/300 batch...
  261/300 batch...
  281/300 batch...

Successfully got 15000 entities


In [3]:
# ─────────────────────────────────────────────
# Step 3：Collect all property IDs and batch-fetch their labels
# ─────────────────────────────────────────────
all_prop_ids = set()
for qid, entity in all_entities.items():
    for prop_id, statements in entity.get('claims', {}).items():
        all_prop_ids.add(prop_id)
        for stmt in statements:
            for q_prop in stmt.get('qualifiers', {}).keys():
                all_prop_ids.add(q_prop)
            for ref in stmt.get('references', []):
                for ref_prop in ref.get('snaks', {}).keys():
                    all_prop_ids.add(ref_prop)

print(f"\nFound {len(all_prop_ids)} unique properties. Fetching label...")

def get_labels_batch(pid_list):
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbgetentities",
        "ids": "|".join(pid_list),
        "format": "json",
        "languages": "en",
        "props": "labels"
    }
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=60)
        return resp.json()
    except Exception as e:
        print(f"  Request failed: {e}")
        return None

prop_labels = {}
prop_list = list(all_prop_ids)
prop_batches = [prop_list[i:i+50] for i in range(0, len(prop_list), 50)]

for i, batch in enumerate(prop_batches):
    if i % 20 == 0:
        print(f"  Property batch {i+1}/{len(prop_batches)}...")
    data = get_labels_batch(batch)
    if data and 'entities' in data:
        for pid, ent in data['entities'].items():
            lbl = ent.get('labels', {}).get('en', {}).get('value', pid)
            prop_labels[pid] = lbl
    time.sleep(0.2)

print("Property label(main property + qualifier + reference) fetched successfully.")


Found 2013 unique properties. Fetching label...
  Property batch 1/41...
  Property batch 21/41...
  Property batch 41/41...
Property label(main property + qualifier + reference) fetched successfully.


In [4]:
# ─────────────────────────────────────────────
# Step 4: Collect all QIDs from values and fetch their labels
# ─────────────────────────────────────────────
print("Collecting all QIDs...")
value_qids = set()

for qid, entity in all_entities.items():
    for prop_id, statements in entity.get('claims', {}).items():
        for stmt in statements:
            dv = stmt.get('mainsnak', {}).get('datavalue', {})
            if dv.get('type') == 'wikibase-entityid':
                value_qids.add(dv['value'].get('id', ''))
            
            for q_snaks in stmt.get('qualifiers', {}).values():
                for q_snak in q_snaks:
                    dv = q_snak.get('datavalue', {})
                    if dv.get('type') == 'wikibase-entityid':
                        value_qids.add(dv['value'].get('id', ''))
            
            for ref in stmt.get('references', []):
                for ref_snaks in ref.get('snaks', {}).values():
                    for ref_snak in ref_snaks:
                        dv = ref_snak.get('datavalue', {})
                        if dv.get('type') == 'wikibase-entityid':
                            value_qids.add(dv['value'].get('id', ''))

value_qids.discard('')
print(f"{len(value_qids)} QIDs. Fetching labels...")

value_qid_list = list(value_qids)
value_batches = [value_qid_list[i:i+50] for i in range(0, len(value_qid_list), 50)]
value_labels = {}  # QID → label

for i, batch in enumerate(value_batches):
    if i % 20 == 0:
        print(f"  Value label batch {i+1}/{len(value_batches)}...")
    data = get_labels_batch(batch)
    if data and 'entities' in data:
        for vid, ent in data['entities'].items():
            lbl = ent.get('labels', {}).get('en', {}).get('value', vid)
            value_labels[vid] = lbl
    time.sleep(0.2)

print("Fetching QID labels is finished.")

60402 QIDs. Fetching labels...
  Value label batch 1/1209...
  Value label batch 21/1209...
  Value label batch 41/1209...
  Value label batch 61/1209...
  Value label batch 81/1209...
  Value label batch 101/1209...
  Value label batch 121/1209...
  Value label batch 141/1209...
  Value label batch 161/1209...
  Value label batch 181/1209...
  Value label batch 201/1209...
  Value label batch 221/1209...
  Value label batch 241/1209...
  Value label batch 261/1209...
  Value label batch 281/1209...
  Value label batch 301/1209...
  Value label batch 321/1209...
  Value label batch 341/1209...
  Value label batch 361/1209...
  Value label batch 381/1209...
  Value label batch 401/1209...
  Value label batch 421/1209...
  Value label batch 441/1209...
  Value label batch 461/1209...
  Value label batch 481/1209...
  Value label batch 501/1209...
  Value label batch 521/1209...
  Value label batch 541/1209...
  Value label batch 561/1209...
  Value label batch 581/1209...
  Value label b

In [5]:
# ─────────────────────────────────────────────
# Step 5: Determine the max qualifier count
# ─────────────────────────────────────────────
print("Calculating the max qualifier count...")
MAX_QUAL = 0
for qid, entity in all_entities.items():
    for prop_id, statements in entity.get('claims', {}).items():
        for stmt in statements:
            qual_count = sum(len(v) for v in stmt.get('qualifiers', {}).values())
            if qual_count > MAX_QUAL:
                MAX_QUAL = qual_count
print(f"The max qualifier count: {MAX_QUAL}")

Calculating the max qualifier count...
The max qualifier count: 19


In [6]:
# ─────────────────────────────────────────────
# Step 6: Fetching all data
# ─────────────────────────────────────────────
def get_snak_value(snak):
    datavalue = snak.get('datavalue', {})
    vtype = datavalue.get('type', '')
    val = datavalue.get('value', '')

    if vtype == 'wikibase-entityid':
        qid = val.get('id', '')
        label = value_labels.get(qid) or prop_labels.get(qid, qid)
        return qid, label
    elif vtype == 'string':
        return val, val
    elif vtype == 'monolingualtext':
        return val.get('text', ''), val.get('text', '')
    elif vtype == 'quantity':
        return val.get('amount', ''), val.get('amount', '')
    elif vtype == 'time':
        return val.get('time', ''), val.get('time', '')
    elif vtype == 'globecoordinate':
        return f"{val.get('latitude','')},{val.get('longitude','')}", \
               f"{val.get('latitude','')},{val.get('longitude','')}"
    else:
        return str(val), str(val)

rows = []

for qid, entity in all_entities.items():
    entity_label = entity.get('labels', {}).get('en', {}).get('value', qid)
    
    for prop_id, statements in entity.get('claims', {}).items():
        prop_label = prop_labels.get(prop_id, prop_id)
        total_statements = len(statements)
        
        for stmt_idx, stmt in enumerate(statements):
            row = {}
            row['QID']               = qid
            row['Entity_Label']      = entity_label
            row['Property_ID']       = prop_id
            row['Property_Label']    = prop_label
            row['Total_Statements']  = total_statements
            row['Statement_Number']  = stmt_idx + 1
            row['Rank']              = stmt.get('rank', '')
            
            # Main value
            mainsnak = stmt.get('mainsnak', {})
            row['Value_Type'] = mainsnak.get('datatype', '')
            main_val, main_label = get_snak_value(mainsnak)
            row['Main_Value'] = main_label
            
            # Qualifiers
            qualifiers = stmt.get('qualifiers', {})
            qual_order = stmt.get('qualifiers-order', list(qualifiers.keys()))
            
            qual_list = []
            for q_prop in qual_order:
                for q_snak in qualifiers.get(q_prop, []):
                    q_val, q_label = get_snak_value(q_snak)
                    qual_list.append({
                        'id':    q_prop,
                        'label': prop_labels.get(q_prop, q_prop),
                        'value': q_label
                    })
            
            row['Has_Qualifiers']   = 'Yes' if qual_list else 'No'
            row['Qualifier_Count']  = len(qual_list)
            
            for i in range(MAX_QUAL):
                if i < len(qual_list):
                    row[f'Qualifier_ID({i+1})']    = qual_list[i]['id']
                    row[f'Qualifier_Label({i+1})']  = qual_list[i]['label']
                    row[f'Qualifier_Value({i+1})']  = qual_list[i]['value']
                else:
                    row[f'Qualifier_ID({i+1})']    = ''
                    row[f'Qualifier_Label({i+1})']  = ''
                    row[f'Qualifier_Value({i+1})']  = ''
            
            # Reference
            references = stmt.get('references', [])
            row['Has_References']  = 'Yes' if references else 'No'
            row['Reference_Count'] = len(references)
            
            ref_parts = []
            for ref in references:
                for ref_prop, ref_snaks in ref.get('snaks', {}).items():
                    for ref_snak in ref_snaks:
                        ref_val, ref_val_label = get_snak_value(ref_snak)
                        ref_parts.append(
                            f"{ref_prop}="
                            f"{prop_labels.get(ref_prop, ref_prop)}="
                            f"{ref_val_label}"
                        )
            row['Reference_Labels'] = ' | '.join(ref_parts)
            
            rows.append(row)

# ─────────────────────────────────────────────
# Step 7: Save
# ─────────────────────────────────────────────
print(f"\n{len(rows)} records in total，saving...")
result_df = pd.DataFrame(rows)
result_df.to_excel('15000politician_statements.xlsx', index=False)
print(f"Done! Saved to 15000politician_statements.xlsx")
print(f"Total rows:   {len(result_df)}")
print(f"Entity count:   {result_df['QID'].nunique()}")
print(f"Property types: {result_df['Property_ID'].nunique()}")


274560 records in total，saving...
Done! Saved to 15000politician_statements.xlsx
Total rows:   274560
Entity count:   14998
Property types: 1787
